In [ ]:
# ============================================================
#  Configuration — N개 런 비교 분석
# ============================================================
RESULTS_DIR = "results"

# 비교할 런 디렉토리 이름 목록 (수동 지정 또는 자동 탐색)
# 수동: RUN_NAMES = ["20260407_185538_G_...", "20260407_185821_H_G_..."]
# 자동: RUN_NAMES = None  → RESULTS_DIR 내 모든 런 로드
RUN_NAMES = None

# 필터 (RUN_NAMES=None일 때 적용)
FILTER_DATE   = "20260407"   # 날짜 prefix (None이면 전체)
FILTER_GPU    = None          # GPU 이름 포함 (예: "H100", None이면 전체)
FILTER_MODEL  = None          # 모델 이름 포함 (예: "Qwen2.5-1.5B", None이면 전체)
# ============================================================

import json, warnings
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.figsize': (14, 5), 'font.size': 11})

results_path = Path(RESULTS_DIR)

def load_run(run_dir):
    """Load a single run directory."""
    run = {'dir': run_dir, 'name': run_dir.name}
    for mode in ('hybrid', 'gpu_only'):
        jf = run_dir / f'{mode}.json'
        if jf.exists():
            with open(jf) as f: run['bench'] = json.load(f)
            run['mode'] = mode
            break
    if 'bench' not in run: return None
    si_path = run_dir / 'system_info.json'
    if si_path.exists():
        with open(si_path) as f: run['sysinfo'] = json.load(f)
    else:
        run['sysinfo'] = {}
    for mtype in ('gpu', 'cpu'):
        csv_path = run_dir / f"{run['mode']}_monitor_{mtype}.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            if 'elapsed_s' in df.columns:
                df['elapsed_s'] = df['elapsed_s'] - df['elapsed_s'].iloc[0]
            run[f'{mtype}_csv'] = df
        else:
            run[f'{mtype}_csv'] = None
    return run

def make_label(run):
    hc = run.get('sysinfo', {}).get('hybrid_config', {})
    mode = run.get('mode', '?')
    model = run['bench'].get('model_id', '?').split('/')[-1]
    if mode == 'gpu_only':
        return f'GPU-only | {model}'
    s = hc.get('routing_strategy', '?')
    p = hc.get('routing_priority', '?')
    seqs = hc.get('cpu_max_seqs', '?')
    routing = 'RR' if s == 'round-robin' else f'{s}({p})'
    return f'Hybrid {routing} seqs={seqs} | {model}'

# Load runs
if RUN_NAMES:
    run_dirs = [results_path / n for n in RUN_NAMES]
else:
    run_dirs = sorted([d for d in results_path.iterdir() if d.is_dir()])
    if FILTER_DATE:
        run_dirs = [d for d in run_dirs if d.name.startswith(FILTER_DATE)]
    if FILTER_GPU:
        run_dirs = [d for d in run_dirs if FILTER_GPU in d.name]
    if FILTER_MODEL:
        run_dirs = [d for d in run_dirs if FILTER_MODEL in d.name]

runs = []
for d in run_dirs:
    r = load_run(d)
    if r: runs.append(r)

print(f'Loaded {len(runs)} runs')
for i, r in enumerate(runs):
    print(f'  [{i}] {make_label(r)}  —  {r["name"]}')

In [ ]:
# ============================================================
#  Section 1: Summary Table
# ============================================================

rows = []
for i, r in enumerate(runs):
    b = r['bench']
    si = r.get('sysinfo', {})
    gpu_devs = si.get('gpu', {}).get('devices', [{}])
    gpu_name = gpu_devs[0].get('name', '-') if gpu_devs else '-'

    rows.append({
        '#': i,
        'Label': make_label(r),
        'Wall (s)': round(b.get('wall_time_s', 0), 1) if b.get('wall_time_s') else '-',
        'Duration (s)': round(b.get('duration', 0), 1),
        'Req/s': round(b.get('request_throughput', 0), 2),
        'Tok/s': round(b.get('output_throughput', 0), 0),
        'TTFT (ms)': round(b.get('mean_ttft_ms', 0), 0),
        'TPOT (ms)': round(b.get('mean_tpot_ms', 0), 2),
        'Completed': f"{b.get('completed','?')}/{b.get('num_prompts','?')}",
        'GPU': gpu_name,
    })

summary_df = pd.DataFrame(rows).set_index('#')

def color_rows(row):
    label = row.get('Label', '')
    if 'GPU-only' in label:
        return ['background-color: #fffde7'] * len(row)
    elif 'cpu-first' in label:
        return ['background-color: #e3f2fd'] * len(row)
    elif 'gpu-first' in label:
        return ['background-color: #e8f5e9'] * len(row)
    elif 'RR' in label:
        return ['background-color: #fce4ec'] * len(row)
    return ['background-color: #f5f5f5'] * len(row)

display(summary_df.style.apply(color_rows, axis=1)
        .set_table_styles([
            {'selector':'th','props':[('background-color','#2d4a7a'),('color','white'),('font-size','12px'),('padding','5px 10px')]},
        ])
        .set_caption('<b>All Runs Summary</b> — yellow=GPU-only, blue=cpu-first, green=gpu-first, pink=RR'))

In [ ]:
# ============================================================
#  Section 2: vs [0] Comparison
# ============================================================

if len(runs) >= 2:
    METRICS = [
        ('wall_time_s',            'Wall Time (s)',     'lower',  '.1f'),
        ('request_throughput',     'Req/s',             'higher', '.2f'),
        ('output_throughput',      'Tok/s',             'higher', '.0f'),
        ('duration',               'Duration (s)',      'lower',  '.1f'),
        ('mean_ttft_ms',           'Mean TTFT (ms)',    'lower',  '.1f'),
        ('p99_ttft_ms',            'P99 TTFT (ms)',     'lower',  '.1f'),
        ('mean_tpot_ms',           'Mean TPOT (ms)',    'lower',  '.2f'),
        ('p99_tpot_ms',            'P99 TPOT (ms)',     'lower',  '.2f'),
    ]

    base = runs[0]['bench']
    print(f'Base [0]: {make_label(runs[0])}\n')

    for i in range(1, len(runs)):
        comp = runs[i]['bench']
        diff_rows = []
        for key, label, direction, fmt in METRICS:
            bv = base.get(key)
            cv = comp.get(key)
            if bv is None or cv is None or bv == 0: continue
            diff_pct = (cv - bv) / abs(bv) * 100
            if direction == 'higher':
                marker = '▲' if diff_pct > 1 else ('▼' if diff_pct < -1 else '~')
            else:
                marker = '▼' if diff_pct > 1 else ('▲' if diff_pct < -1 else '~')
            diff_rows.append({
                'Metric': label,
                '[0]': format(bv, fmt),
                f'[{i}]': format(cv, fmt),
                'Diff': f'{diff_pct:+.1f}%',
                '': marker,
            })
        if diff_rows:
            df = pd.DataFrame(diff_rows).set_index('Metric')
            def highlight_diff(val):
                if '▲' in str(val): return 'color: green; font-weight: bold'
                if '▼' in str(val): return 'color: red; font-weight: bold'
                return ''
            display(df.style.applymap(highlight_diff, subset=[''])
                    .set_caption(f'<b>[{i}] {make_label(runs[i])} vs [0]</b>'))
else:
    print('Need >= 2 runs for comparison')

In [ ]:
# ============================================================
#  Section 3: Throughput Bar Chart
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

labels_short = [f'[{i}]' for i in range(len(runs))]
colors = []
for r in runs:
    l = make_label(r)
    if 'GPU-only' in l: colors.append('#FFC107')
    elif 'cpu-first' in l: colors.append('#2196F3')
    elif 'gpu-first' in l: colors.append('#4CAF50')
    elif 'RR' in l: colors.append('#E91E63')
    else: colors.append('#9E9E9E')

# Req/s
ax = axes[0]
vals = [r['bench'].get('request_throughput', 0) for r in runs]
ax.bar(labels_short, vals, color=colors)
ax.set_title('Request Throughput (req/s)')
ax.set_ylabel('req/s')
for j, v in enumerate(vals): ax.text(j, v, f'{v:.1f}', ha='center', va='bottom', fontsize=9)

# Tok/s
ax = axes[1]
vals = [r['bench'].get('output_throughput', 0) for r in runs]
ax.bar(labels_short, vals, color=colors)
ax.set_title('Output Throughput (tok/s)')
ax.set_ylabel('tok/s')
for j, v in enumerate(vals): ax.text(j, v, f'{v:.0f}', ha='center', va='bottom', fontsize=9)

# Mean TTFT
ax = axes[2]
vals = [r['bench'].get('mean_ttft_ms', 0) for r in runs]
ax.bar(labels_short, vals, color=colors)
ax.set_title('Mean TTFT (ms)')
ax.set_ylabel('ms')
for j, v in enumerate(vals): ax.text(j, v, f'{v:.0f}', ha='center', va='bottom', fontsize=9)

# Legend
patches = []
for i, r in enumerate(runs):
    patches.append(mpatches.Patch(color=colors[i], label=f'[{i}] {make_label(r)[:40]}'))
fig.legend(handles=patches, loc='lower center', ncol=min(3, len(runs)), fontsize=9,
           bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  Section 4: GPU Utilization Comparison
# ============================================================

fig, ax = plt.subplots(figsize=(16, 5))
for i, r in enumerate(runs):
    csv = r.get('gpu_csv')
    if csv is not None and 'gpu_avg_util_pct' in csv.columns:
        ax.plot(csv['elapsed_s'], csv['gpu_avg_util_pct'],
                label=f'[{i}] {make_label(r)[:35]}', alpha=0.8, lw=1.5)
ax.set_xlabel('Time (s)'); ax.set_ylabel('GPU Avg Utilization %')
ax.set_title('GPU Utilization — All Runs')
ax.set_ylim(-5, 105); ax.grid(True, alpha=0.3)
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
#  Section 5: CPU Utilization Comparison
# ============================================================

fig, ax = plt.subplots(figsize=(16, 5))
for i, r in enumerate(runs):
    csv = r.get('cpu_csv')
    if csv is not None and 'cpu_avg_util_pct' in csv.columns:
        ax.plot(csv['elapsed_s'], csv['cpu_avg_util_pct'],
                label=f'[{i}] {make_label(r)[:35]}', alpha=0.8, lw=1.5)
ax.set_xlabel('Time (s)'); ax.set_ylabel('CPU Avg Utilization %')
ax.set_title('CPU Utilization — All Runs')
ax.set_ylim(-5, 105); ax.grid(True, alpha=0.3)
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
#  Section 6: GPU Power Comparison
# ============================================================

fig, ax = plt.subplots(figsize=(16, 5))
for i, r in enumerate(runs):
    csv = r.get('gpu_csv')
    if csv is not None and 'gpu_avg_power_w' in csv.columns:
        ax.plot(csv['elapsed_s'], csv['gpu_avg_power_w'],
                label=f'[{i}] {make_label(r)[:35]}', alpha=0.8, lw=1.5)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Power (W)')
ax.set_title('GPU Power — All Runs')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
#  Section 7: Per-Run CPU Heatmaps
# ============================================================

for i, r in enumerate(runs):
    csv = r.get('cpu_csv')
    if csv is None: continue
    core_cols = [c for c in csv.columns if c.startswith('core') and c.endswith('_util_pct')]
    if not core_cols: continue

    fig, ax = plt.subplots(figsize=(16, 3))
    data = csv[core_cols].values.T
    im = ax.imshow(data, aspect='auto', cmap='YlOrRd', vmin=0, vmax=100,
                   extent=[csv['elapsed_s'].iloc[0], csv['elapsed_s'].iloc[-1],
                           len(core_cols)-0.5, -0.5])
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Core')
    ax.set_title(f'[{i}] {make_label(r)[:50]} — CPU Core Heatmap')
    plt.colorbar(im, ax=ax, label='%', shrink=0.8)
    plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
#  Section 8: System Info per Run
# ============================================================

tbl_style = [
    {'selector':'th','props':[('background-color','#2d4a7a'),('color','white'),
                              ('font-size','12px'),('padding','5px 10px')]},
    {'selector':'tr:nth-child(even)','props':[('background-color','#f5f8ff')]},
    {'selector':'td','props':[('font-size','12px'),('padding','4px 10px')]},
]

for i, r in enumerate(runs):
    si = r.get('sysinfo', {})
    hc = si.get('hybrid_config', {})
    cpu = si.get('cpu', {})
    gpu_devs = si.get('gpu', {}).get('devices', [{}])
    gpu_name = gpu_devs[0].get('name', '-') if gpu_devs else '-'
    sw = si.get('software', {})

    info_rows = [
        ('Mode', r.get('mode', '?')),
        ('CPU', cpu.get('model_name', '-')),
        ('GPU', gpu_name),
        ('vLLM', sw.get('vllm_version', '-')),
    ]
    if hc:
        s = hc.get('routing_strategy', '?')
        p = hc.get('routing_priority', '?')
        info_rows += [
            ('Routing', s if s == 'round-robin' else f'{s} ({p})'),
            ('CPU Max Seqs', hc.get('cpu_max_seqs', '?')),
            ('NUMA Aware', hc.get('numa_aware', '?')),
        ]

    df = pd.DataFrame(info_rows, columns=['Item','Value']).set_index('Item')
    display(df.style.set_table_styles(tbl_style)
            .set_caption(f'<b>[{i}] {make_label(r)}</b>'))